Understand what customers buy, purchasing patterns, basket size trends, department and product wise performance and first vs repeat basket composition across their shopping journey.

**Basket overview: 
Before any segmentation, we establish the baseline basket metrics across all
orders.**

In [17]:
basket_KPIs = spark.sql("""
SELECT
    COUNT(DISTINCT op.order_id) AS Orders,
    COUNT(op.product_id) AS Items_Sold,
    ROUND(COUNT(op.product_id) * 1.0 / COUNT(DISTINCT op.order_id), 2) 
    As Avg_Basket_Size,
    ROUND(MIN(basket_size), 0) AS Min_Basket_Size,
    ROUND(MAX(basket_size), 0) AS Max_Basket_Size
FROM order_products op
JOIN (
    SELECT order_id, COUNT(*) AS basket_size
    FROM order_products
    GROUP BY order_id
) bs ON op.order_id = bs.order_id
""").toPandas()

basket_KPIs.style \
    .format({'Orders': '{:,}','Items_Sold': '{:,}','Avg_Basket_Size': '{:.2f}',
    'Min_Basket_Size': '{:,.0f}','Max_Basket_Size': '{:,.0f}'}).hide(axis='index')

StatementMeta(, c772a087-6f9e-4ee6-be8d-5a5b75e3e830, 21, Finished, Available, Finished, False)

Orders,Items_Sold,Avg_Basket_Size,Min_Basket_Size,Max_Basket_Size
"3,346,083","33,819,106",10.11,1,145


_**3,346,083 orders** generated **33,819,106 items sold**  giving an **average basket size of 10.11 items** per order. Baskets range from a minimum of **1 item** to maximum of **145 items** in a single order_

_The 10-item average basket is a commercially significant anchor - it means
customers come to Instacart for a meaningful shopping, not a top-up trip_

**What is the average basket size and how does it vary by order sequence?**

In [26]:
basket_by_sequence = spark.sql("""
SELECT
    o.order_number AS Order_Sequence,
    COUNT(DISTINCT o.order_id) AS Orders,
    ROUND(COUNT(op.product_id) * 1.0/COUNT(DISTINCT o.order_id), 2) 
    AS Avg_Basket_Size,
    ROUND(AVG(op.reordered) * 100, 1) AS Reorder_Rate
FROM orders o
JOIN order_products op
ON o.order_id = op.order_id
GROUP BY o.order_number 
ORDER BY o.order_number
""").toPandas()

display(basket_by_sequence.head(5).style.format({
    'Orders': '{:,}','Avg_Basket_Size': '{:.2f}','Reorder_Rate': '{:.1f}%'
    }).hide(axis='index')
)
display(basket_by_sequence.tail(5).style.format({
    'Orders': '{:,}', 'Avg_Basket_Size': '{:.2f}','Reorder_Rate': '{:.1f}%'
    }).hide(axis='index')
)

StatementMeta(, c772a087-6f9e-4ee6-be8d-5a5b75e3e830, 94, Finished, Available, Finished, False)

Order_Sequence,Orders,Avg_Basket_Size,Reorder_Rate
1,"206,209",10.08,0.0%
2,"206,209",9.93,27.2%
3,"206,209",9.94,38.6%
4,"197,523",9.97,45.4%
5,"175,072",10.01,50.3%


Order_Sequence,Orders,Avg_Basket_Size,Reorder_Rate
96,"1,569",9.06,84.4%
97,"1,501",9.12,85.3%
98,"1,452",9.06,85.3%
99,"1,405",9.03,84.2%
100,867,8.79,86.0%


**Which departments and aisles drive the most volume and the highest reorder rates?**

In [5]:
dept_performance = spark.sql("""
SELECT
    d.department AS Department,
    COUNT(op.product_id) AS Items_Sold,
    COUNT(DISTINCT op.product_id) AS Unique_Products,
    ROUND(AVG(op.reordered) * 100, 1) AS Reorder_Rate,
    ROUND(COUNT(op.product_id) * 100.0 / SUM(COUNT(op.product_id)) 
    OVER(), 2) AS Total_Volume
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY Items_Sold DESC
""").toPandas()

dept_performance.style.format({'Items_Sold': '{:,}','Unique_Products': '{:,}',
'Reorder_Rate': '{:.1f}%', 'Total_Volume': '{:.1f}%'}).hide(axis='index')

StatementMeta(, 3daf7e6b-54b0-4523-b6ba-a04c9cc41a9a, 8, Finished, Available, Finished, False)

Department,Items_Sold,Unique_Products,Reorder_Rate,Total_Volume
produce,"9,888,378","1,684",65.1%,29.2%
dairy eggs,"5,631,067","3,449",67.0%,16.6%
snacks,"3,006,412","6,264",57.4%,8.9%
beverages,"2,804,175","4,364",65.4%,8.3%
frozen,"2,336,858","4,007",54.3%,6.9%
pantry,"1,956,819","5,370",34.7%,5.8%
bakery,"1,225,181","1,516",62.8%,3.6%
canned goods,"1,114,857","2,092",45.9%,3.3%
deli,"1,095,540","1,322",60.8%,3.2%
dry goods pasta,"905,340","1,858",46.2%,2.7%


_Produce was the largest department with **9,888,378 items sold**, representing approximately **29.2%** of all purchased items and a **65.1%** reorder rate. \
Dairy & Eggs ranked second with **5,631,067 items sold** and recorded the highest reorder rate **(67.0%)** among the major departments, highlighting strong customer loyalty.\
The top four departments together accounted for the majority of purchased items, demonstrating that everyday grocery products drive overall sales volume._

**What is in the typical first basket vs the typical repeat basket?**

First order (order_number = 1) customers try new things or first order 
Repeat orders (order_number ≥ 10) habit mode they settle into categories.

In [1]:
dept_performance = spark.sql("""
SELECT
    d.department,
    COUNT(CASE WHEN o.order_number = 1 THEN op.product_id END) AS `1st Order`,
    COUNT(CASE WHEN o.order_number BETWEEN 2 AND 10 THEN op.product_id END) AS `Orders 2-10`,
    COUNT(CASE WHEN o.order_number BETWEEN 11 AND 20 THEN op.product_id END) AS `Orders 11-20`,
    COUNT(CASE WHEN o.order_number >= 21 THEN op.product_id END) AS `Orders 21+`
FROM orders o
JOIN order_products op ON o.order_id = op.order_id
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY `Orders 21+` DESC
""").toPandas()

dept_performance.style.format({'1st Order': '{:,}','Orders 2-10': '{:,}',
'Orders 11-20': '{:,}','Orders 21+': '{:,}'}).hide(axis='index')

StatementMeta(, 1623c044-f948-443f-9454-271669742dd3, 3, Finished, Available, Finished, False)

department,1st Order,Orders 2-10,Orders 11-20,Orders 21+
produce,"584,384","4,087,831","2,159,660","3,056,503"
dairy eggs,"333,030","2,328,820","1,240,766","1,728,451"
snacks,"180,909","1,269,406","663,718","892,379"
beverages,"167,618","1,190,937","620,718","824,902"
frozen,"158,323","1,071,651","509,605","597,279"
pantry,"128,268","862,896","418,387","547,268"
bakery,"75,541","521,047","267,308","361,285"
deli,"70,099","479,087","240,164","306,190"
canned goods,"73,766","494,061","241,983","305,047"
dry goods pasta,"60,982","408,270","195,486","240,602"


_Other major departments, including Snacks **(180,909 → 892,379 items)**, Beverages **(167,618 → 824,902 items)**, and Frozen **(158,323 → 597,279 items)**, experienced substantial increases after customers first purchase, suggesting that shopping baskets become more diverse as customers continue using the platform. \
Across nearly all departments, purchases were highest during orders 2–10, indicating that customers rapidly expand their shopping behaviour after their initial order before settling into more stable purchasing patterns._

**Products most often added to cart first by volume?**

In [2]:
cart_position = spark.sql("""
SELECT
    p.product_name,
    d.department,
    a.aisle,
    COUNT(*) AS total_appearances,
    ROUND(SUM(CASE WHEN op.add_to_cart_order = 1 THEN 1 ELSE 0 END) 
    * 100.0 / COUNT(*), 1) AS first_in_cart,
    ROUND(SUM(CASE WHEN op.add_to_cart_order = 2 THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1) AS second_in_cart
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN aisles a ON p.aisle_id = a.aisle_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY
    p.product_name,
    d.department,
    a.aisle
ORDER BY total_appearances DESC
LIMIT 20
""").toPandas()

cart_position.style.format({'total_appearances': '{:,}',
'first_in_cart': '{:.1f}%', 'second_in_cart': '{:.1f}%'}).hide(axis='index')

StatementMeta(, e2adcc90-5177-4419-8f40-f4237f9a6c8a, 5, Finished, Available, Finished, False)

product_name,department,aisle,total_appearances,first_in_cart,second_in_cart
Banana,produce,fresh fruits,"491,291",23.5%,17.0%
Bag of Organic Bananas,produce,fresh fruits,"394,930",21.0%,17.0%
Organic Strawberries,produce,fresh fruits,"275,577",10.5%,11.2%
Organic Baby Spinach,produce,packaged vegetables fruits,"251,705",9.7%,10.4%
Organic Hass Avocado,produce,fresh fruits,"220,877",11.3%,12.4%
Organic Avocado,produce,fresh fruits,"184,224",12.7%,13.3%
Large Lemon,produce,fresh fruits,"160,792",8.0%,9.3%
Strawberries,produce,fresh fruits,"149,445",11.4%,11.8%
Limes,produce,fresh fruits,"146,660",6.9%,8.2%
Organic Whole Milk,dairy eggs,milk,"142,813",22.5%,15.1%


_All of the top seven products belong to the Produce department, demonstrating that fresh fruits and vegetables frequently initiate customer shopping sessions and strongly influence overall basket composition._